# 02 — Data Cleaning

## GulfMart Retail Inventory Analytics

This notebook resolves the data-quality issues identified in
`01_data_profiling.ipynb` (Section 9, Findings Summary) and intentionally
planted by `inject_data_quality_issues()` during data generation.

Cleaning logic lives in `src/data_cleaning.py`, not in this notebook
directly -- this notebook loads raw data, applies that logic, verifies the
result with before/after comparisons, re-runs the full validation suite
from `src/data_validation.py` to prove the fixes actually worked, and
saves the cleaned tables to `data/cleaned/`.

**What gets cleaned:**

| Table | Issue | Strategy |
|---|---|---|
| `dim_customer` | Missing `city` | Impute `"Unknown"` |
| `dim_supplier` | Missing `supplier_name` | Impute a supplier_id-traceable placeholder |
| `dim_store` | Inconsistent `city` text formatting | Normalize (strip + title case) |
| `fact_sales` | Duplicate transactions | Drop duplicates, keep first occurrence |

**What does NOT get cleaned, on purpose:**

- `dim_product` -- never targeted by data-quality injection, passed through unchanged.
- `fact_inventory` -- had zero missing values, duplicates, or text issues in
  profiling (data-quality injection intentionally excludes it, to avoid
  contradicting its continuity/reconciliation validation). Loaded here
  only so the full table set can be re-validated together; not modified.
- Products with zero recorded sales, and inventory rows with very high
  coverage-days -- legitimate business characteristics identified in
  profiling, not data-quality defects. Left untouched.


## 1. Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> project root
SRC_DIR = PROJECT_ROOT / "src"
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import pandas as pd

from data_generation_config import CLEANED_DATA_DIR
from data_cleaning import clean_generated_dataset, save_cleaned_datasets
from data_validation import validate_generated_dataset

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data: {DATA_RAW_DIR}")
print(f"Cleaned data output: {CLEANED_DATA_DIR}")


Project root: d:\GitHub\retail-inventory-analytics
Raw data: d:\GitHub\retail-inventory-analytics\data\raw
Cleaned data output: D:\GitHub\retail-inventory-analytics\data\cleaned


## 2. Load Raw Data

`fact_inventory` is loaded here too (it will not be modified) so it can
be passed into the re-validation step later, alongside the cleaned
tables -- `validate_generated_dataset()` expects all six tables together.

In [2]:
dim_product = pd.read_csv(DATA_RAW_DIR / "dim_product.csv")
dim_store = pd.read_csv(DATA_RAW_DIR / "dim_store.csv")
dim_customer = pd.read_csv(DATA_RAW_DIR / "dim_customer.csv")
dim_supplier = pd.read_csv(DATA_RAW_DIR / "dim_supplier.csv")
fact_sales = pd.read_csv(DATA_RAW_DIR / "fact_sales.csv")
fact_inventory = pd.read_csv(DATA_RAW_DIR / "fact_inventory.csv.gz")

print("Raw tables loaded.")
print()
print("Known issues before cleaning:")
print(f"  dim_customer missing city:              {dim_customer['city'].isna().sum():,}")
print(f"  dim_supplier missing supplier_name:      {dim_supplier['supplier_name'].isna().sum():,}")
print(f"  fact_sales duplicate transaction_id:     {fact_sales['transaction_id'].duplicated().sum():,}")
print(f"  dim_store city values (visual check):    {dim_store['city'].tolist()}")


Raw tables loaded.

Known issues before cleaning:
  dim_customer missing city:              4
  dim_supplier missing supplier_name:      1
  fact_sales duplicate transaction_id:     125
  dim_store city values (visual check):    ['Abha', 'Dammam', 'Riyadh', 'Abha', 'Makkah', 'Makkah', 'Abha', 'Tabuk', 'Abha', 'Tabuk', 'Abha', 'Riyadh', 'Dammam', 'Makkah', 'Madinah', 'Jeddah', 'Makkah', 'Khobar', 'Abha', 'Tabuk']


## 3. Run the Cleaning Pipeline

`clean_generated_dataset()` runs all four cleaning steps and prints a
report for each. `dim_product` and `fact_inventory` are intentionally
excluded from cleaning -- see the table above.

In [3]:
cleaned = clean_generated_dataset(
    dim_product,
    dim_store,
    dim_customer,
    dim_supplier,
    fact_sales,
)

dim_product_clean = cleaned["dim_product"]
dim_store_clean = cleaned["dim_store"]
dim_customer_clean = cleaned["dim_customer"]
dim_supplier_clean = cleaned["dim_supplier"]
fact_sales_clean = cleaned["fact_sales"]



GulfMart Retail Inventory Dataset Cleaning

Cleaning dim_customer
Filled 4 missing 'city' values with 'Unknown'.

Cleaning dim_supplier
Filled 1 missing 'supplier_name' values with a supplier_id-traceable placeholder.

Cleaning dim_store
No inconsistent 'city' formatting found -- nothing to normalize.

Cleaning fact_sales
Removed 125 duplicate transaction(s). 125,125 -> 125,000 rows.

CLEANING SUMMARY
dim_customer -- city values filled:        4
dim_supplier -- supplier_name values filled: 1
dim_store    -- city values normalized:      0
fact_sales   -- duplicate rows removed:      125


## 4. Before / After Comparison

Show the exact rows that changed, not just counts, so the cleaning is
verifiable rather than taken on faith.

In [4]:
imputed_customer_mask = dim_customer["city"].isna()

customer_comparison = pd.DataFrame({
    "customer_id": dim_customer.loc[imputed_customer_mask, "customer_id"],
    "city_before": dim_customer.loc[imputed_customer_mask, "city"],
    "city_after": dim_customer_clean.loc[imputed_customer_mask, "city"],
})

print("dim_customer -- imputed rows:")
print(customer_comparison.to_string(index=False) if not customer_comparison.empty else "(none)")


dim_customer -- imputed rows:
customer_id city_before city_after
  CUST01761         NaN    Unknown
  CUST01881         NaN    Unknown
  CUST02241         NaN    Unknown
  CUST03005         NaN    Unknown


In [5]:
imputed_supplier_mask = dim_supplier["supplier_name"].isna()

supplier_comparison = pd.DataFrame({
    "supplier_id": dim_supplier.loc[imputed_supplier_mask, "supplier_id"],
    "supplier_name_before": dim_supplier.loc[imputed_supplier_mask, "supplier_name"],
    "supplier_name_after": dim_supplier_clean.loc[imputed_supplier_mask, "supplier_name"],
})

print("dim_supplier -- imputed rows:")
print(supplier_comparison.to_string(index=False) if not supplier_comparison.empty else "(none)")


dim_supplier -- imputed rows:
supplier_id supplier_name_before       supplier_name_after
     SUP004                  NaN Unknown Supplier (SUP004)


In [6]:
changed_store_mask = dim_store["city"] != dim_store_clean["city"]

store_comparison = pd.DataFrame({
    "store_id": dim_store.loc[changed_store_mask, "store_id"],
    "city_before": dim_store.loc[changed_store_mask, "city"],
    "city_after": dim_store_clean.loc[changed_store_mask, "city"],
})

print("dim_store -- normalized rows:")
print(store_comparison.to_string(index=False) if not store_comparison.empty else "(none this run)")


dim_store -- normalized rows:
(none this run)


In [7]:
print(f"fact_sales row count before cleaning: {len(fact_sales):,}")
print(f"fact_sales row count after cleaning:  {len(fact_sales_clean):,}")
print(f"Rows removed:                         {len(fact_sales) - len(fact_sales_clean):,}")


fact_sales row count before cleaning: 125,125
fact_sales row count after cleaning:  125,000
Rows removed:                         125


## 5. Save Cleaned Datasets to `data/cleaned/`

In [8]:
save_cleaned_datasets(cleaned, CLEANED_DATA_DIR)



Saving cleaned datasets
Saved dim_product            500 rows  ->  dim_product.csv  (0.07 MB)
Saved dim_store               20 rows  ->  dim_store.csv  (0.00 MB)
Saved dim_customer         5,000 rows  ->  dim_customer.csv  (0.36 MB)
Saved dim_supplier            30 rows  ->  dim_supplier.csv  (0.00 MB)
Saved fact_sales         125,000 rows  ->  fact_sales.csv  (26.50 MB)

Output directory: D:\GitHub\retail-inventory-analytics\data\cleaned


## 6. Re-Validate the Cleaned Dataset

Runs the exact same `validate_generated_dataset()` suite used in
`01_data_profiling.ipynb`, now against the cleaned tables (plus the
unchanged `fact_inventory`). `fact_sales` should flip from **FAIL** to
**PASS**, since the duplicate transactions that caused it are now
removed. `inventory_reconciliation` and `lost_sales` will still show
**REVIEW** -- expected and unrelated to cleaning; see the note in
`01_data_profiling.ipynb`.

In [9]:
validation_results_after_cleaning = validate_generated_dataset(
    fact_sales_clean,
    fact_inventory,
    dim_product_clean,
    dim_store_clean,
    dim_customer_clean,
    dim_supplier_clean,
)



GulfMart Retail Inventory Dataset Validation

Dataset Summary
Products:       500
Stores:         20
Customers:      5,000
Suppliers:      30
Sales rows:     125,000
Inventory rows: 10,960,000

Product Dimension Validation
PASS: Product Dimension key completeness - NULL product_id values: 0
PASS: Product Dimension key uniqueness - Duplicate product_id values: 0

Store Dimension Validation
PASS: Store Dimension key completeness - NULL store_id values: 0
PASS: Store Dimension key uniqueness - Duplicate store_id values: 0

Customer Dimension Validation
PASS: Customer Dimension key completeness - NULL customer_id values: 0
PASS: Customer Dimension key uniqueness - Duplicate customer_id values: 0

Supplier Dimension Validation
PASS: Supplier Dimension key completeness - NULL supplier_id values: 0
PASS: Supplier Dimension key uniqueness - Duplicate supplier_id values: 0

Fact Sales Validation
PASS: Required columns - Missing columns: []
PASS: Transaction ID uniqueness - Duplicate transactio

## 7. Findings Summary

Update this section after each run.

**Cleaning actions taken (update counts to match the run above):**

- `dim_customer.city` -- missing values imputed with `"Unknown"`
- `dim_supplier.supplier_name` -- missing values imputed with a
  supplier_id-traceable placeholder (e.g. `"Unknown Supplier (SUP014)"`)
- `dim_store.city` -- text formatting normalized (strip + title case),
  where applicable
- `fact_sales` -- duplicate transactions removed, keeping the first
  occurrence of each

**Validation result:** `fact_sales` should now show **PASS** (was FAIL
in `01_data_profiling.ipynb`, before cleaning). All other checks should
match the profiling notebook's result: 15 PASS, 2 REVIEW
(`inventory_reconciliation`, `lost_sales` -- architectural, not
applicable to the current inventory model).

**Left deliberately unchanged:**

- `dim_product`, `fact_inventory` -- no planted issues to clean.
- Products with zero recorded sales, and inventory rows with very high
  coverage-days -- legitimate business findings, not data-quality
  defects; carried forward into later analysis notebooks instead.

**Next step:** `03_inventory_kpis.ipynb` builds on the cleaned tables in
`data/cleaned/` (dimensions + `fact_sales`) together with `fact_inventory`
from `data/raw/`.